In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras.layers import (
    TextVectorization,
    Embedding,
    Dense,
    LayerNormalization,
    MultiHeadAttention,
    Dropout
)

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices("GPU"))

TensorFlow version: 2.21.0
GPU available: []


In [3]:
data = pd.read_pickle("../merged_training.pkl")

print("Shape:", data.shape)
print(data.head())

Shape: (416809, 2)
                                                     text emotions
27383   i feel awful about it too because it s my job ...  sadness
110083                              im alone i feel awful  sadness
140764  ive probably mentioned this before but i reall...      joy
100071           i was feeling a little low few days back  sadness
2837    i beleive that i am much more sensitive to oth...     love


## Clean the text corpus

In [4]:
data = data.drop_duplicates(subset="text").reset_index(drop=True)

texts = data["text"].astype(str)

print("Unique texts:", len(texts))
print("First text:", texts.iloc[0])

Unique texts: 393822
First text: i feel awful about it too because it s my job to get him in a position to succeed and it just didn t happen here


#### Select the training corpus

In [5]:
texts = texts.sample(
    n=100000,
    random_state=42
).reset_index(drop=True)

print("Texts for Transformer:", len(texts))
print(texts.head())

Texts for Transformer: 100000
0    i grew up feeling like an outsider hated and b...
1    i feel disturbed with the view that the poor a...
2    i was probably feeling very relaxed somewhere ...
3    i am so thrilled because i feel like this will...
4    im ready feel i liked it other people complime...
Name: text, dtype: str


## Train/validation split

In [6]:
from sklearn.model_selection import train_test_split

train_texts, val_texts = train_test_split(
    texts,
    test_size=0.10,
    random_state=42
)

print("Training texts:", len(train_texts))
print("Validation texts:", len(val_texts))

Training texts: 90000
Validation texts: 10000


## Training texts: 90000
## Validation texts: 10000

####  We'll use a 20,000-token vocabulary and a 64-token context window for our first CPU-friendly model.

In [7]:
from tensorflow.keras.layers import TextVectorization

MAX_TOKENS = 20000
SEQ_LEN = 64

vectorizer = TextVectorization(
    max_tokens=MAX_TOKENS,
    output_mode="int",
    output_sequence_length=SEQ_LEN + 1
)

vectorizer.adapt(train_texts)

print("Vocabulary size:", len(vectorizer.get_vocabulary()))

Vocabulary size: 20000


#### Convert one sentence into token IDs

In [8]:
sample_text = train_texts.iloc[0]

sample_tokens = vectorizer(tf.constant([sample_text]))[0]

print("Original text:")
print(sample_text)

print("\nToken IDs:")
print(sample_tokens.numpy())

print("\nShape:")
print(sample_tokens.shape)

Original text:
i distinctly remember feeling very unhappy with myself that day even though i ran several miles because i felt like i was not as fit or fast as i should be

Token IDs:
[   2 3715  196    8   47  506   25   53    9   97   74  137    2 1597
 1095 1370   38    2  153   14    2   21   26   29 1035   35 1087   29
    2  130   27    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0]

Shape:
(65,)


### Create input and target

In [9]:
decoder_input = sample_tokens[:-1]
decoder_target = sample_tokens[1:]

print("Decoder input:")
print(decoder_input.numpy())

print("\nDecoder target:")
print(decoder_target.numpy())

print("\nInput shape:", decoder_input.shape)
print("Target shape:", decoder_target.shape)

Decoder input:
[   2 3715  196    8   47  506   25   53    9   97   74  137    2 1597
 1095 1370   38    2  153   14    2   21   26   29 1035   35 1087   29
    2  130   27    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0]

Decoder target:
[3715  196    8   47  506   25   53    9   97   74  137    2 1597 1095
 1370   38    2  153   14    2   21   26   29 1035   35 1087   29    2
  130   27    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0]

Input shape: (64,)
Target shape: (64,)


## Token + positional embeddings

In [10]:
VOCAB_SIZE = 20000
EMBED_DIM = 128
MAX_LEN = 64

token_embedding = Embedding(
    input_dim=VOCAB_SIZE,
    output_dim=EMBED_DIM
)

position_embedding = Embedding(
    input_dim=MAX_LEN,
    output_dim=EMBED_DIM
)

# Add batch dimension
decoder_input_batch = tf.expand_dims(decoder_input, axis=0)

# Token embeddings
token_vectors = token_embedding(decoder_input_batch)

# Position IDs
positions = tf.range(start=0, limit=MAX_LEN, delta=1)

# Position embeddings
position_vectors = position_embedding(positions)

# Combine token + position information
decoder_x = token_vectors + position_vectors

print("Token embedding shape:", token_vectors.shape)
print("Position embedding shape:", position_vectors.shape)
print("Decoder input shape:", decoder_x.shape)

Token embedding shape: (1, 64, 128)
Position embedding shape: (64, 128)
Decoder input shape: (1, 64, 128)


#### Causal self-attention

In [11]:
NUM_HEADS = 4
KEY_DIM = 32

causal_attention = MultiHeadAttention(
    num_heads=NUM_HEADS,
    key_dim=KEY_DIM
)

attention_output = causal_attention(
    query=decoder_x,
    key=decoder_x,
    value=decoder_x,
    use_causal_mask=True
)

print("Input shape:", decoder_x.shape)
print("Attention output shape:", attention_output.shape)

Input shape: (1, 64, 128)
Attention output shape: (1, 64, 128)


#### Add & Norm

In [12]:
norm_1 = LayerNormalization()

residual_1 = decoder_x + attention_output

normalized_1 = norm_1(residual_1)

print("Residual shape:", residual_1.shape)
print("Normalized shape:", normalized_1.shape)

Residual shape: (1, 64, 128)
Normalized shape: (1, 64, 128)


#### Feed-Forward Network

In [13]:
ffn_dense_1 = Dense(
    256,
    activation="relu"
)

ffn_dense_2 = Dense(
    128
)

ffn_hidden = ffn_dense_1(normalized_1)
ffn_output = ffn_dense_2(ffn_hidden)

print("FFN hidden shape:", ffn_hidden.shape)
print("FFN output shape:", ffn_output.shape)

FFN hidden shape: (1, 64, 256)
FFN output shape: (1, 64, 128)


#### Second Add & Norm

In [14]:
norm_2 = LayerNormalization()

residual_2 = normalized_1 + ffn_output

decoder_output = norm_2(residual_2)

print("Second residual shape:", residual_2.shape)
print("Decoder output shape:", decoder_output.shape)

Second residual shape: (1, 64, 128)
Decoder output shape: (1, 64, 128)


#### Project decoder output to vocabulary

In [15]:
output_projection = Dense(VOCAB_SIZE)

logits = output_projection(decoder_output)

print("Decoder output shape:", decoder_output.shape)
print("Logits shape:", logits.shape)

Decoder output shape: (1, 64, 128)
Logits shape: (1, 64, 20000)


### Convert logits to probabilities

#### For each position, we'll convert the 20,000 logits into probabilities

In [16]:
probabilities = tf.nn.softmax(logits, axis=-1)

print("Probability shape:", probabilities.shape)

print("\nProbability sum at first position:")
print(tf.reduce_sum(probabilities[0, 0]))

Probability shape: (1, 64, 20000)

Probability sum at first position:
tf.Tensor(1.0000001, shape=(), dtype=float32)


#### Calculate next-token loss

In [17]:
loss_function = tf.keras.losses.SparseCategoricalCrossentropy(
    from_logits=True,
    reduction="none"
)

loss_per_token = loss_function(
    decoder_target,
    logits[0]
)

# Ignore padding tokens
mask = tf.cast(decoder_target != 0, tf.float32)

masked_loss = loss_per_token * mask

loss = tf.reduce_sum(masked_loss) / tf.reduce_sum(mask)

print("Loss:", loss.numpy())

Loss: 9.924585


### Turn our decoder block into a reusable class

In [18]:
class DecoderOnlyBlock(tf.keras.layers.Layer):

    def __init__(self, embed_dim, num_heads, ff_dim, dropout_rate=0.1):
        super().__init__()

        self.attention = MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim // num_heads
        )

        self.ffn_dense_1 = Dense(
            ff_dim,
            activation="relu"
        )

        self.ffn_dense_2 = Dense(
            embed_dim
        )

        self.norm_1 = LayerNormalization()
        self.norm_2 = LayerNormalization()

        self.dropout_1 = Dropout(dropout_rate)
        self.dropout_2 = Dropout(dropout_rate)

    def call(self, inputs, training=False):

        # Causal self-attention
        attention_output = self.attention(
            query=inputs,
            key=inputs,
            value=inputs,
            use_causal_mask=True
        )

        attention_output = self.dropout_1(
            attention_output,
            training=training
        )

        # Add & Norm
        residual_1 = inputs + attention_output
        normalized_1 = self.norm_1(residual_1)

        # Feed-Forward Network
        ffn_hidden = self.ffn_dense_1(normalized_1)
        ffn_output = self.ffn_dense_2(ffn_hidden)

        ffn_output = self.dropout_2(
            ffn_output,
            training=training
        )

        # Add & Norm
        residual_2 = normalized_1 + ffn_output
        output = self.norm_2(residual_2)

        return output

In [19]:
decoder_block = DecoderOnlyBlock(
    embed_dim=128,
    num_heads=4,
    ff_dim=256
)

test_output = decoder_block(
    decoder_x,
    training=False
)

print("Decoder block output shape:", test_output.shape)

Decoder block output shape: (1, 64, 128)


### Stack decoder blocks

In [20]:
NUM_BLOCKS = 2

decoder_blocks = [
    DecoderOnlyBlock(
        embed_dim=128,
        num_heads=4,
        ff_dim=256,
        dropout_rate=0.1
    )
    for _ in range(NUM_BLOCKS)
]

x = decoder_x

for i, block in enumerate(decoder_blocks, start=1):
    x = block(x, training=False)
    print(f"After Decoder Block {i}:", x.shape)

After Decoder Block 1: (1, 64, 128)
After Decoder Block 2: (1, 64, 128)


#### Final LayerNorm + Vocabulary Projection

In [21]:
final_norm = LayerNormalization()

x = final_norm(x)

output_projection = Dense(VOCAB_SIZE)

logits = output_projection(x)

print("Final normalized shape:", x.shape)
print("Logits shape:", logits.shape)

Final normalized shape: (1, 64, 128)
Logits shape: (1, 64, 20000)


### Build the complete mode

In [22]:
class DecoderOnlyTransformer(tf.keras.Model):

    def __init__(
        self,
        vocab_size,
        max_len,
        embed_dim,
        num_heads,
        ff_dim,
        num_blocks,
        dropout_rate=0.1
    ):
        super().__init__()

        self.max_len = max_len

        self.token_embedding = Embedding(
            input_dim=vocab_size,
            output_dim=embed_dim
        )

        self.position_embedding = Embedding(
            input_dim=max_len,
            output_dim=embed_dim
        )

        self.decoder_blocks = [
            DecoderOnlyBlock(
                embed_dim=embed_dim,
                num_heads=num_heads,
                ff_dim=ff_dim,
                dropout_rate=dropout_rate
            )
            for _ in range(num_blocks)
        ]

        self.final_norm = LayerNormalization()

        self.output_projection = Dense(vocab_size)

    def call(self, inputs, training=False):

        # Token embeddings
        token_vectors = self.token_embedding(inputs)

        # Position IDs
        positions = tf.range(
            start=0,
            limit=tf.shape(inputs)[1]
        )

        # Position embeddings
        position_vectors = self.position_embedding(positions)

        # Token + position
        x = token_vectors + position_vectors

        # Decoder blocks
        for block in self.decoder_blocks:
            x = block(
                x,
                training=training
            )

        # Final normalization
        x = self.final_norm(x)

        # Next-token logits
        logits = self.output_projection(x)

        return logits

In [23]:
model = DecoderOnlyTransformer(
    vocab_size=VOCAB_SIZE,
    max_len=MAX_LEN,
    embed_dim=128,
    num_heads=4,
    ff_dim=256,
    num_blocks=2,
    dropout_rate=0.1
)

test_logits = model(
    tf.expand_dims(decoder_input, axis=0),
    training=False
)

print("Model output shape:", test_logits.shape)

Model output shape: (1, 64, 20000)


#### That means for each of the 64 positions, it predicts a probability distribution over 20,000 possible next tokens.

### Create a text-to-token datase

In [24]:
BATCH_SIZE = 32

def prepare_text(text):
    tokens = vectorizer(text)
    
    inputs = tokens[:, :-1]
    targets = tokens[:, 1:]
    
    return inputs, targets


train_ds = (
    tf.data.Dataset.from_tensor_slices(train_texts.values)
    .batch(BATCH_SIZE)
    .map(prepare_text, num_parallel_calls=tf.data.AUTOTUNE)
    .prefetch(tf.data.AUTOTUNE)
)

val_ds = (
    tf.data.Dataset.from_tensor_slices(val_texts.values)
    .batch(BATCH_SIZE)
    .map(prepare_text, num_parallel_calls=tf.data.AUTOTUNE)
    .prefetch(tf.data.AUTOTUNE)
)

print("Dataset created.")

Dataset created.


#### Perfect ✅ The dataset pipeline was created successfully

In [25]:
for batch_inputs, batch_targets in train_ds.take(1):
    print("Input shape:", batch_inputs.shape)
    print("Target shape:", batch_targets.shape)

Input shape: (32, 64)
Target shape: (32, 64)


#### Add padding mask + compile the model

In [26]:
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(
    from_logits=True
)

optimizer = tf.keras.optimizers.Adam(
    learning_rate=1e-4
)

def add_mask(inputs, targets):
    mask = tf.cast(targets != 0, tf.float32)
    return inputs, targets, mask


train_ds_masked = train_ds.map(
    add_mask,
    num_parallel_calls=tf.data.AUTOTUNE
)

val_ds_masked = val_ds.map(
    add_mask,
    num_parallel_calls=tf.data.AUTOTUNE
)

model.compile(
    optimizer=optimizer,
    loss=loss_fn
)

print("Model compiled successfully.")

Model compiled successfully.


##### Then verify one batch:

In [27]:
for batch_inputs, batch_targets, batch_mask in train_ds_masked.take(1):
    print("Input:", batch_inputs.shape)
    print("Target:", batch_targets.shape)
    print("Mask:", batch_mask.shape)

Input: (32, 64)
Target: (32, 64)
Mask: (32, 64)


### Replace the decoder block with this version

In [28]:
class DecoderOnlyBlock(tf.keras.layers.Layer):

    def __init__(self, embed_dim, num_heads, ff_dim, dropout_rate=0.1):
        super().__init__()

        self.attention = MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim // num_heads
        )

        self.ffn_dense_1 = Dense(
            ff_dim,
            activation="relu"
        )

        self.ffn_dense_2 = Dense(
            embed_dim
        )

        self.norm_1 = LayerNormalization()
        self.norm_2 = LayerNormalization()

        self.dropout_1 = Dropout(dropout_rate)
        self.dropout_2 = Dropout(dropout_rate)

    def call(self, inputs, attention_mask=None, training=False):

        attention_output = self.attention(
            query=inputs,
            key=inputs,
            value=inputs,
            attention_mask=attention_mask
        )

        attention_output = self.dropout_1(
            attention_output,
            training=training
        )

        residual_1 = inputs + attention_output
        normalized_1 = self.norm_1(residual_1)

        ffn_hidden = self.ffn_dense_1(normalized_1)
        ffn_output = self.ffn_dense_2(ffn_hidden)

        ffn_output = self.dropout_2(
            ffn_output,
            training=training
        )

        residual_2 = normalized_1 + ffn_output
        output = self.norm_2(residual_2)

        return output

#### Update the complete model

In [29]:
class DecoderOnlyTransformer(tf.keras.Model):

    def __init__(
        self,
        vocab_size,
        max_len,
        embed_dim,
        num_heads,
        ff_dim,
        num_blocks,
        dropout_rate=0.1
    ):
        super().__init__()

        self.max_len = max_len

        self.token_embedding = Embedding(
            input_dim=vocab_size,
            output_dim=embed_dim
        )

        self.position_embedding = Embedding(
            input_dim=max_len,
            output_dim=embed_dim
        )

        self.decoder_blocks = [
            DecoderOnlyBlock(
                embed_dim=embed_dim,
                num_heads=num_heads,
                ff_dim=ff_dim,
                dropout_rate=dropout_rate
            )
            for _ in range(num_blocks)
        ]

        self.final_norm = LayerNormalization()

        self.output_projection = Dense(vocab_size)

    def call(self, inputs, training=False):

        batch_size = tf.shape(inputs)[0]
        seq_len = tf.shape(inputs)[1]

        # Padding mask
        padding_mask = tf.not_equal(inputs, 0)
        padding_mask = padding_mask[:, tf.newaxis, :]

        # Causal mask
        causal_mask = tf.linalg.band_part(
            tf.ones(
                (seq_len, seq_len),
                dtype=tf.bool
            ),
            -1,
            0
        )

        causal_mask = causal_mask[tf.newaxis, :, :]

        # Combined mask
        attention_mask = tf.logical_and(
            causal_mask,
            padding_mask
        )

        # Token embeddings
        token_vectors = self.token_embedding(inputs)

        # Position embeddings
        positions = tf.range(
            start=0,
            limit=seq_len
        )

        position_vectors = self.position_embedding(positions)

        # Token + position
        x = token_vectors + position_vectors

        # Decoder blocks
        for block in self.decoder_blocks:
            x = block(
                x,
                attention_mask=attention_mask,
                training=training
            )

        # Final normalization
        x = self.final_norm(x)

        # Vocabulary logits
        logits = self.output_projection(x)

        return logits

#### Recreate and test the model

In [30]:
model = DecoderOnlyTransformer(
    vocab_size=VOCAB_SIZE,
    max_len=MAX_LEN,
    embed_dim=128,
    num_heads=4,
    ff_dim=256,
    num_blocks=2,
    dropout_rate=0.1
)

test_logits = model(
    batch_inputs,
    training=False
)

print("Output shape:", test_logits.shape)

Output shape: (32, 64, 20000)


### Compile again and run a small training test

In [31]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-4
    ),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(
        from_logits=True
    )
)

print("Compiled successfully")

Compiled successfully


In [32]:
history = model.fit(
    train_ds_masked,
    validation_data=val_ds_masked,
    epochs=1,
    steps_per_epoch=100
)

c:\dl_env\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


100/100 ━━━━━━━━━━━━━━━━━━━━ 81s 777ms/step - loss: 2.6659 - val_loss: 2.3646


#### Save this checkpoint

In [33]:
model.save_weights("decoder_transformer.weights.h5")

print("Weights saved")

Weights saved
